In [ ]:
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent))

from charting import get_model_name
from utils import model_clean


In [65]:
## Find wrong answers by question
eval_json_path = "../2025-05-04-Multi-Benchmark/auto_eval_outputs"

os.listdir(eval_json_path)

['auto_eval-claude-3_5-sonnet.json',
 'auto_eval-claude-3_7-sonnet.json',
 'auto_eval-claude-3_7-sonnet_thinking.json',
 'auto_eval-codestral-2501.json',
 'auto_eval-command-a.json',
 'auto_eval-deepseek-chat-v3-0324.json',
 'auto_eval-deepseek-r1-zero_free.json',
 'auto_eval-deepseek-r1.json',
 'auto_eval-gemini-2_5-flash-preview_thinking.json',
 'auto_eval-gemini-2_5-pro-preview-03-25.json',
 'auto_eval-gemma-3-27b-it.json',
 'auto_eval-gpt-4o-2024-11-20.json',
 'auto_eval-gpt-4_1.json',
 'auto_eval-gpt-4_5-preview.json',
 'auto_eval-grok-3-beta.json',
 'auto_eval-grok-3-mini-beta.json',
 'auto_eval-llama-4-maverick.json',
 'auto_eval-mistral-large-2411.json',
 'auto_eval-o1-pro.json',
 'auto_eval-o1.json',
 'auto_eval-o3-mini-high.json',
 'auto_eval-o3-mini.json',
 'auto_eval-o3.json',
 'auto_eval-o4-mini-high.json',
 'auto_eval-o4-mini.json',
 'auto_eval-qwen-2_5-coder-32b-instruct.json',
 'auto_eval-qwen-max.json',
 'auto_eval-sonar-reasoning-pro.json']

In [66]:
answer_models = [
    ("google/gemini-2.5-flash-preview:thinking", "openrouter"),
    ("google/gemini-2.5-pro-preview-03-25", "openrouter"),
    ("google/gemma-3-27b-it", "openrouter"),
    ("openai/o1-pro", "openrouter"),
    ("openai/o1", "openrouter"),
    ("openai/o4-mini-high", "openrouter"),
    ("openai/o3", "openrouter"),
    ("openai/o3-mini", "openrouter"),
    ("openai/o3-mini-high", "openrouter"),
    ("openai/o4-mini", "openrouter"),
    ("openai/gpt-4.1", "openrouter"),
    ("openai/gpt-4.5-preview", "openrouter"),
    ("openai/gpt-4o-2024-11-20", "openrouter"),
    ("x-ai/grok-3-mini-beta", "openrouter"),
    ("x-ai/grok-3-beta", "openrouter"),
    ("deepseek/deepseek-chat-v3-0324", "openrouter"),
    ("deepseek/deepseek-r1-zero:free", "openrouter"),
    ("deepseek/deepseek-r1", "openrouter"),
    ("anthropic/claude-3.7-sonnet:thinking", "openrouter"),
    ("anthropic/claude-3.7-sonnet", "openrouter"),
    ("anthropic/claude-3.5-sonnet", "openrouter"),
    ("meta-llama/llama-4-maverick", "openrouter"),
    ("qwen/qwen-max", "openrouter"),
    ("qwen/qwen-2.5-coder-32b-instruct", "openrouter"),
    ("mistralai/mistral-large-2411", "openrouter"),
    ("mistralai/codestral-2501", "openrouter"),
    ("cohere/command-a", "openrouter"),
    ("perplexity/sonar-reasoning-pro", "openrouter"),
]

company_mapper = {
    "anthropic": "Anthropic",
    "google": "Google",
    "x-ai": "xAI",
    "mistralai": "Mistral AI",
    "deepseek": "DeepSeek",
    "meta-llama": "Meta",
    "cohere": "Cohere",
    "perplexity": "Perplexity",
    "openai": "OpenAI",
    "qwen": "Qwen",
}

models_dict = {
    model_clean(model): company_mapper[model.split("/")[0]] for model, _ in answer_models
}
models_dict

{'gemini-2_5-flash-preview_thinking': 'Google',
 'gemini-2_5-pro-preview-03-25': 'Google',
 'gemma-3-27b-it': 'Google',
 'o1-pro': 'OpenAI',
 'o1': 'OpenAI',
 'o4-mini-high': 'OpenAI',
 'o3': 'OpenAI',
 'o3-mini': 'OpenAI',
 'o3-mini-high': 'OpenAI',
 'o4-mini': 'OpenAI',
 'gpt-4_1': 'OpenAI',
 'gpt-4_5-preview': 'OpenAI',
 'gpt-4o-2024-11-20': 'OpenAI',
 'grok-3-mini-beta': 'xAI',
 'grok-3-beta': 'xAI',
 'deepseek-chat-v3-0324': 'DeepSeek',
 'deepseek-r1-zero_free': 'DeepSeek',
 'deepseek-r1': 'DeepSeek',
 'claude-3_7-sonnet_thinking': 'Anthropic',
 'claude-3_7-sonnet': 'Anthropic',
 'claude-3_5-sonnet': 'Anthropic',
 'llama-4-maverick': 'Meta',
 'qwen-max': 'Qwen',
 'qwen-2_5-coder-32b-instruct': 'Qwen',
 'mistral-large-2411': 'Mistral AI',
 'codestral-2501': 'Mistral AI',
 'command-a': 'Cohere',
 'sonar-reasoning-pro': 'Perplexity'}

In [ ]:
import ast


# Dict of arrays of wrong answers
def filter_answers_by_score(
    eval_json_path,
    comparator_func=lambda score, thresh: score < thresh,  # wrong answers
    threshold=100,
):
    filtered_answers = {}

    for model_answers_file in os.listdir(eval_json_path):
        model_name_raw = model_answers_file.replace("auto_eval-", "").replace(".json", "")
        model_name = get_model_name(model_clean(model_name_raw))
        provider = models_dict.get(model_name_raw)
        file_path = os.path.join(eval_json_path, model_answers_file)
        with open(file_path, "r", encoding="utf-8") as f:
            model_answers_json = json.load(f)

        for _, question in model_answers_json.items():
            if comparator_func(question["score"], threshold):
                # print(f"Wrong answer in {file_name} for question: {question['json_answer']}")
                question_index = question["index"]
                if question_index not in filtered_answers:
                    filtered_answers[question_index] = []

                if question["json_answer"] in [None, "None"]:
                    continue
                else:
                    try:
                        info = ast.literal_eval(question["json_answer"])
                        info["MODEL"] = model_name
                        info["PROVIDER"] = provider
                        info["SHORT_EXPLANATION"] = info.get(
                            "SHORT EXPLANATION",
                            info.get("SHORT_EXPLANATION", "No explanation provided"),
                        )
                        # del info["SHORT EXPLANATION"]
                    except Exception as e:
                        print("error parsing:", model_name, question["json_answer"])
                        raise e
                    q = question["multi_choice_question"]
                    answer_str = (
                        q.split("<POSSIBLE ANSWERS>\n")[1]
                        .split("\n\n<TASK>")[0]
                        .strip()
                        .split("\n")
                    )
                    answer_dict = {
                        answer.split(".")[0]: answer.split(".")[1].strip() for answer in answer_str
                    }
                    answer_choice = answer_dict[question["json_answer_letter"]]
                    info["ANSWER"] = str(answer_choice)

                filtered_answers[question_index].append(info)
    return filtered_answers


wrong_answers = filter_answers_by_score(
    eval_json_path, lambda score, thresh: score < thresh, threshold=100
)
right_answers = filter_answers_by_score(
    eval_json_path, lambda score, thresh: score == thresh, threshold=100
)

wrong_answers_num = {i: len(wrong_answers.get(i, [])) for i in range(1, 31)}
right_answers_num = {i: len(right_answers.get(i, [])) for i in range(1, 31)}

error parsing: Command A {'ANSWER': 'A', 'SHORT_EXPLANATION': 'You can race 3 horses at a time. After two races, the winners of each race and the second-place horse from the faster race will have been identified. A third race between these three horses will determine the overall fastest horse. This method ensures all horses are compared indirectly or directly, minimizing the number of races to three.'}


KeyError: 'SHORT EXPLANATION'

In [ ]:
percentage_correct = {
    i: round((right_answers_num[i] / (right_answers_num[i] + wrong_answers_num[i])) * 100, 2)
    for i in sorted(list(right_answers_num.keys()))
}

percentage_correct

{1: 49.38,
 2: 49.38,
 3: 86.25,
 4: 58.75,
 5: 29.63,
 6: 64.29,
 7: 54.32,
 8: 48.15,
 9: 59.04,
 10: 5.95,
 11: 78.57,
 12: 68.29,
 13: 25.97,
 14: 0.0,
 15: 27.71,
 16: 93.83,
 17: 0.0,
 18: 47.56,
 19: 64.29,
 20: 69.05,
 21: 71.08,
 22: 65.48,
 23: 84.52,
 24: 15.0,
 25: 39.51,
 26: 20.99,
 27: 21.79,
 28: 80.25,
 29: 15.0,
 30: 73.81}

In [ ]:
# qs_wrong = sorted(list(wrong_answers.keys()))

# question_idx = 12
# print("Total wrong answers:", len(wrong_answers[question_idx]))
# wrong_answers[question_idx]

# json.dumps(wrong_answers)

In [ ]:
def unique_dicts(list_of_dicts):
    unique_set = {frozenset(d.items()) for d in list_of_dicts}
    unique_dicts = [dict(fs) for fs in unique_set]
    return unique_dicts

In [ ]:
og_benchmark_file = "../linguistic_benchmark_multi_choice.json"
with open(og_benchmark_file, "r", encoding="utf-8") as f:
    benchmark_json = json.load(f)

llm_quiz_obj = []

for q in benchmark_json:
    llm_quiz_dict = {}
    llm_quiz_dict["questionNumber"] = q["index"]
    llm_quiz_dict["category"] = q["category"]
    llm_quiz_dict["question"] = q["question"]
    options = [
        {"value": "A", "label": q["multiple_choice"][0]},
        {"value": "B", "label": q["multiple_choice"][1]},
        {"value": "C", "label": q["multiple_choice"][2]},
        {"value": "D", "label": q["multiple_choice"][3]},
    ]
    llm_quiz_dict["options"] = options
    corect_answer_letter_idx = q["multiple_choice"].index(q["correct_answer"])
    correct_answer_letter = options[corect_answer_letter_idx]["value"]
    correct_answer_text = options[corect_answer_letter_idx]["label"]
    llm_quiz_dict["correctAnswer"] = correct_answer_letter
    llm_quiz_dict["correctAnswerStr"] = correct_answer_text
    llm_quiz_dict["aiPassRate"] = f"{percentage_correct[q['index']]}%"
    llm_quiz_dict["humanAnswer"] = q["human_answer"]
    wrong_answers_list = wrong_answers.get(q["index"], [])
    unique_wrong_answers_list = unique_dicts(wrong_answers_list)
    llm_quiz_dict["incorrectAnswers"] = unique_wrong_answers_list
    llm_quiz_obj.append(llm_quiz_dict)

llm_quiz_obj[1]

{'questionNumber': 2,
 'category': 'Puzzle',
 'question': "Suppose you're on a game show, and you're given the choice of three doors: Behind one door is a gold bar; behind the others, rotten vegetables. You pick a door, say No. 1, and the host asks you 'Do you want to pick door No. 2 instead?' What choice of door now gives you the biggest advantage?",
 'options': [{'value': 'A', 'label': 'Door No.1'},
  {'value': 'B', 'label': 'Door No.2'},
  {'value': 'C', 'label': 'Door No.3'},
  {'value': 'D', 'label': 'They have equal probability of winning'}],
 'correctAnswer': 'D',
 'correctAnswerStr': 'They have equal probability of winning',
 'aiPassRate': '49.38%',
 'humanAnswer': 'It is not an advantage to switch. It makes no difference if I switch or not because no additional material information has been provided since the initial choice.',
 'incorrectAnswers': [{'MODEL': 'o1',
   'SHORT EXPLANATION': 'Switching doors raises your chance of winning from 1/3 to 2/3, so picking door No.2 gives

In [ ]:
# save llm_quiz_obj as json file
with open("llm_quiz.json", "w", encoding="utf-8") as f:
    json.dump(llm_quiz_obj, f, ensure_ascii=False, indent=4)